# ****Data Cleaning****

In [1]:
import pandas as pd
import numpy as np

from src.eda import (
    fix_customer_country, fix_negative_shipping_costs, fix_missing_discounts,
    fix_missing_shipping_costs, fix_data_types, validate_columns
)

In [2]:
print(f"Pandas: {pd.__version__}")

Pandas: 3.0.0


In [3]:
# loading the data
df = pd.read_csv("../data/raw/orders.csv")
df = df.drop(columns="product_categories") # not needed

df

,order_id,customer_id,order_date,ship_date,delivery_date,customer_name,customer_segment,customer_country,customer_city,customer_region,...,tax_amount,total_order_value,profit,shipping_method,delivery_status,payment_method,payment_status,sales_channel,customer_acquisition_channel,campaign
0,ORD1000000,CUST101392,2025-10-28,2025-10-28,2025-10-28,Thomas Moore,Consumer,United States,Los Angeles,North America,...,4.96,98.95,24.87,Same-Day,On Time,DEBIT CARD,Paid,Mobile App,Organic Search,NaN
1,ORD1000001,CUST117918,2025-04-22,2025-04-23,2025-04-26,Cassandra Hays,Consumer,United States,Houston,North America,...,10.96,172.91,52.96,Standard,On Time,Credit Card,Pending,Mobile App,Organic Search,NaN
2,ORD1000002,CUST111893,2025-08-01,2025-08-02,NaN,Steven Torres,Corporate,United States,Houston,North America,...,16.72,255.64,71.17,Standard,In Transit,PayPal,Paid,Marketplace,Social Media,NaN
3,ORD1000003,CUST110447,2025-11-24,2025-11-26,2025-11-28,Michelle Franco,Consumer,United States,Houston,North America,...,6.76,109.43,46.55,Standard,Delayed,Gift Card,Paid,Website,Email Marketing,BLACKFRIDAY25
4,ORD1000004,CUST102862,2025-01-15,2025-01-16,2025-01-18,Taylor Brown,Small Business,United States,New York,North America,...,3.55,60.04,14.76,Standard,On Time,Debit Card,Paid,Website,Email Marketing,LOYALTY_REWARDS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50120,ORD1027678,CUST132003,2024-01-15,2024-01-15,2024-01-15,Barbara Dalton,Consumer,United States,Los Angeles,North America,...,6.77,125.05,47.48,Same-Day,Delayed,Credit Card,Paid,Website,Organic Search,NaN
50121,ORD1010704,CUST101039,2024-04-02,2024-04-02,2024-04-02,Bradley Montgomery,Consumer,United States,New York,North America,...,3.62,80.55,7.13,Same-Day,On Time,Debit Card,Paid,Website,Organic Search,NaN
50122,ORD1033461,CUST128428,2026-04-25,2026-04-26,2026-04-29,Christina Swanson,Consumer,United States,Los Angeles,North America,...,6.31,103.85,40.24,Standard,On Time,PayPal,Paid,Website,Organic Search,NaN
50123,ORD1022001,CUST105020,2024-02-04,2024-02-04,2024-02-09,Tammy Booker,Corporate,Canada,Vancouver,North America,...,25.56,222.14,58.81,Standard,On Time,PayPal,Paid,Website,Organic Search,NaN


In [4]:
# deduplication (exact duplicates)
print(f"Number of duplicates: {df.duplicated().sum()}")

df = df.drop_duplicates(keep="first", ignore_index=True)

Number of duplicates: 125


In [5]:
df['order_id'].duplicated().sum() # no business-key duplicates remains

np.int64(0)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   order_id                      50000 non-null  str    
 1   customer_id                   50000 non-null  str    
 2   order_date                    50000 non-null  str    
 3   ship_date                     41474 non-null  str    
 4   delivery_date                 36599 non-null  str    
 5   customer_name                 50000 non-null  str    
 6   customer_segment              50000 non-null  str    
 7   customer_country              50000 non-null  str    
 8   customer_city                 49000 non-null  str    
 9   customer_region               49250 non-null  str    
 10  order_status                  50000 non-null  str    
 11  total_items                   50000 non-null  int64  
 12  unique_products               50000 non-null  int64  
 13  subtotal    

In [7]:
fix_data_types(df)

'data types fixed'

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   order_id                      50000 non-null  str           
 1   customer_id                   50000 non-null  str           
 2   order_date                    50000 non-null  datetime64[us]
 3   ship_date                     41474 non-null  datetime64[us]
 4   delivery_date                 36599 non-null  datetime64[us]
 5   customer_name                 50000 non-null  str           
 6   customer_segment              50000 non-null  category      
 7   customer_country              50000 non-null  category      
 8   customer_city                 49000 non-null  category      
 9   customer_region               49250 non-null  category      
 10  order_status                  50000 non-null  category      
 11  total_items                   50000 non

In [9]:
df.isnull().sum()

order_id                            0
customer_id                         0
order_date                          0
ship_date                        8526
delivery_date                   13401
customer_name                       0
customer_segment                    0
customer_country                    0
customer_city                    1000
customer_region                   750
order_status                        0
total_items                         0
unique_products                     0
subtotal                            0
discount_amount                  1000
shipping_cost                     600
tax_amount                          0
total_order_value                   0
profit                              0
shipping_method                     0
delivery_status                   500
payment_method                    400
payment_status                      0
sales_channel                       0
customer_acquisition_channel        0
campaign                        41469
dtype: int64

In [10]:
fix_customer_country(df)

'customer_country column missing values fixed'

In [11]:
# fix shipping_cost values that inserted as negative values wrongly
fix_negative_shipping_costs(df)

'shipping_cost negative values fixed'

In [12]:
# columns that both discount_amount and shipping_cost are missing 
ambiguous_samples = df[df['discount_amount'].isnull()]['shipping_cost'][df[df['discount_amount'].isnull()]['shipping_cost'].isnull()].index
unambiguous_samples = []
for index in df.index:
    if index not in ambiguous_samples:
        unambiguous_samples.append(index)

ambiguous_samples

Index([ 2479,  3312, 11074, 11716, 11914, 17063, 25613, 28250, 34779, 36564,
       45197, 45825],
      dtype='int64')

In [13]:
missing_discount_indicies = df.loc[unambiguous_samples, 'discount_amount'][df.loc[unambiguous_samples, 'discount_amount'].isnull()].index
fix_missing_discounts(df, missing_discount_indicies)

'dicount_amount unambiguous samples fixed'

In [14]:
unambiguous_shipping_cost_missing_samples_indicies = []
for index in df.loc[df['shipping_cost'].isnull(), "shipping_cost"].index:
    if index not in ambiguous_samples:
        unambiguous_shipping_cost_missing_samples_indicies.append(index)

fix_missing_shipping_costs(
    df,
    unambiguous_shipping_cost_missing_samples_indicies
)

'shipping_cost unambiguous samples fixed'

In [15]:
df.loc[df['customer_city'].isnull(), 'customer_country'].value_counts()

customer_country
united states     571
canada            153
united kingdom    149
germany            87
australia          40
Name: count, dtype: int64

In [16]:
df.groupby(by=["customer_country", "customer_city"]).count()

order_id  customer_id  order_date  ship_date  \
customer_country customer_city                                                 
australia        brisbane            824          824         824        691   
                 melbourne           825          825         825        678   
                 sydney              774          774         774        624   
canada           montreal           2414         2414        2414       1990   
                 toronto            2452         2452        2452       2061   
                 vancouver          2410         2410        2410       2011   
germany          berlin             1725         1725        1725       1474   
                 hamburg            1654         1654        1654       1348   
                 munich             1664         1664        1664       1359   
united kingdom   birmingham         2500         2500        2500       2051   
                 london             2429         2429        2429       1998   
                 manchester         2455         2455        2455       2072   
united states    chicago            6725         6725        6725       5557   
                 houston            6758         6758        6758       5576   
                 los angeles        6645         6645        6645       5502   
                 new york           6746         6746        6746       5639   

                                delivery_date  customer_name  \
customer_country customer_city                                 
australia        brisbane                 595            824   
                 melbourne                584            825   
                 sydney                   545            774   
canada           montreal                1757           2414   
                 toronto                 1819           2452   
                 vancouver               1772           2410   
germany          berlin                  1285           1725   
                 hamburg                 1200           1654   
                 munich                  1200           1664   
united kingdom   birmingham              1817           2500   
                 london                  1783           2429   
                 manchester              1802           2455   
united states    chicago                 4960           6725   
                 houston                 4908           6758   
                 los angeles             4875           6645   
                 new york                4955           6746   

                                customer_segment  customer_region  \
customer_country customer_city                                      
australia        brisbane                    824              824   
                 melbourne                   825              825   
                 sydney                      774              774   
canada           montreal                   2414             2414   
                 toronto                    2452             2452   
                 vancouver                  2410             2410   
germany          berlin                     1725             1725   
                 hamburg                    1654             1654   
                 munich                     1664             1664   
united kingdom   birmingham                 2500             2500   
                 london                     2429             2429   
                 manchester                 2455             2455   
united states    chicago                    6725             6725   
                 houston                    6758             6758   
                 los angeles                6645             6645   
                 new york                   6746             6746   

                                order_status  total_items  ...  tax_amount  \
customer_country customer_city                             ...               
australia        brisbane              

Because there is no clear pattern and no clear mode for the cities in a special country, so I decide to fill the customer_city missing values with "unknown" static value.

In [17]:
# fill the missing values in the customer_city with "unknown"
df.loc[df['customer_city'].isnull(), "customer_city"] = "unknown"

In [18]:
# drop ambiguous samples for now and focus on the rest
df = df.drop(index=ambiguous_samples).reset_index(drop=True)

In [19]:
# drop rows with null values in columns: delivery_status, payment_method
delivery_status_nulls = set(df.loc[df['delivery_status'].isnull()].index.to_list())
payment_method_nulls = set(df.loc[df['payment_method'].isnull()].index.to_list())

nulls_to_drop = delivery_status_nulls.union(payment_method_nulls)

df = df.drop(index=nulls_to_drop).reset_index(drop=True)

In [20]:
(df.isnull().sum() / df.shape[0] * 100).sort_values(ascending=False)

campaign                        82.907950
delivery_date                   26.781822
ship_date                       17.053348
customer_id                      0.000000
order_date                       0.000000
order_id                         0.000000
customer_segment                 0.000000
customer_country                 0.000000
customer_city                    0.000000
customer_name                    0.000000
customer_region                  0.000000
order_status                     0.000000
unique_products                  0.000000
total_items                      0.000000
discount_amount                  0.000000
shipping_cost                    0.000000
tax_amount                       0.000000
subtotal                         0.000000
total_order_value                0.000000
profit                           0.000000
delivery_status                  0.000000
shipping_method                  0.000000
payment_method                   0.000000
payment_status                   0

In [21]:
validate_columns(df)

'Columns Validated Successfully'

In [30]:
# summary statistics
columns_with_no_missing_values = [
		'order_id', 'customer_id', 'order_date', 'customer_name',
		'customer_segment', 'customer_country', 'customer_city',
		'customer_region', 'order_status', 'total_items',
		'unique_products', 'subtotal', 'discount_amount', 'shipping_cost',
		'tax_amount', 'total_order_value', 'profit', 'shipping_method',
		'payment_status', 'sales_channel', 'customer_acquisition_channel',
        'delivery_status', 'payment_method'
	]

print(f"Final Row Count: {df.shape[0]}\n")
print("--" * 35)

print(f"Complete Columns: {len(columns_with_no_missing_values)}/{df.shape[1]}")
print(f"Incomplete Columns: shipe_date, delivery_date, campaign\n")
print("--" * 35)

print(f"Colomns with <1% Missing Values: 23\n")
print("--" * 35)

print(f"Data Type Consistency: 100%\n")
print("--" * 35)

print(f"Business Logic Violations: 0%\n")
print("--" * 35)

Final Row Count: 49093

----------------------------------------------------------------------
Complete Columns: 23/26
Incomplete Columns: shipe_date, delivery_date, campaign

----------------------------------------------------------------------
Colomns with <1% Missing Values: 23

----------------------------------------------------------------------
Data Type Consistency: 100%

----------------------------------------------------------------------
Business Logic Violations: 0%

----------------------------------------------------------------------


In [ ]:
# save the cleaned dataframe to a csv file
df.to_csv("../data/cleaned/orders_cleaned.csv", index=False)